# WDGRL for Regression Under Covariate Shift

**Colab-ready notebook: data inspection, distribution-shift diagnostics, baseline models, and WDGRL adaptation**

This notebook generates synthetic source/target samples from multivariate Gaussian distributions, predicts outcomes with a supplied Random Forest, compares the source and target distributions, and trains Elastic Net, a neural-network baseline, and a Wasserstein Distance Guided Representation Learning (WDGRL) model.

> **Important: three companion files are required.** The uploaded notebook does not contain the files it reads:
> - `rw_data.npz`: means, covariance matrices, and target-noise scale (`mean_train`, `cov_train`, `mean_test`, `cov_test`, `sig_y`)
> - `rf_model.pickle`: pretrained Random Forest used to create synthetic labels
> - `Optim_hyperpar_rf_gaussian.txt`: dictionary-like text file containing the tuned Elastic Net and neural-network hyperparameters
>
> Upload these files in the first setup cell. They are inputs from the original experiment; this notebook does not recreate them or infer their values.

## How to run in Google Colab
1. Run the setup cell and upload the three files.
2. Run cells from top to bottom in order.
3. If TensorFlow/Keras reports an error around `K.gradients`, `decay`, or graph/eager execution, see the compatibility note in the setup section. The WDGRL implementation below uses legacy Keras graph-mode APIs and may require a TensorFlow/Keras version compatible with the original code.
4. The pickle file must come from a trusted source. Pickle/joblib files can execute code while loading.

## 1. Environment, imports, and reproducibility

The random seed makes generated samples and model initialization more repeatable. Exact results can still vary across hardware and library versions.

In [3]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import ast
import random
import warnings
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from scipy.stats import wasserstein_distance

import joblib
import sklearn
from sklearn.linear_model import ElasticNet
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, Input, LeakyReLU, Layer
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.constraints import MaxNorm
import tensorflow.keras.backend as K
from tensorflow.python.framework.ops import disable_eager_execution

disable_eager_execution()  # Required by the original graph-mode gradient-penalty implementation.
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

print(f"Python packages: NumPy {np.__version__} | scikit-learn {sklearn.__version__} | TensorFlow {tf.__version__}")
print("Note: the WDGRL cells use legacy graph-mode Keras APIs.")

Python packages: NumPy 1.23.5 | scikit-learn 1.1.3 | TensorFlow 2.9.3
Note: the WDGRL cells use legacy graph-mode Keras APIs.


### Upload the experiment assets

Use the upload dialog in Colab. If you already placed the files in the current working directory, you can skip the upload. Keep the filenames exactly as shown.

In [4]:
# Google Colab upload helper; on a local Jupyter environment, upload/copy files manually.
try:
    from google.colab import files
    required_files = {
        "rw_data.npz",
        "rf_model.pickle",
        "Optim_hyperpar_rf_gaussian.txt",
    }
    present = {p.name for p in Path(".").iterdir() if p.is_file()}
    missing = required_files - present
    if missing:
        print("Please upload:", ", ".join(sorted(missing)))
        uploaded = files.upload()
    else:
        print("All required files are already present.")
except ImportError:
    print("Not running in Google Colab. Ensure the three required files are in the notebook's working directory.")

required_files = ["rw_data.npz", "rf_model.pickle", "Optim_hyperpar_rf_gaussian.txt"]
missing_files = [name for name in required_files if not Path(name).exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing required experiment asset(s): " + ", ".join(missing_files) +
        ". Upload them before continuing. Their contents cannot be reconstructed from this notebook alone."
    )

Not running in Google Colab. Ensure the three required files are in the notebook's working directory.


FileNotFoundError: Missing required experiment asset(s): rw_data.npz, rf_model.pickle, Optim_hyperpar_rf_gaussian.txt. Upload them before continuing. Their contents cannot be reconstructed from this notebook alone.

## 2. Load the pretrained label-generating model

The Random Forest `f_hat` is not trained in this notebook. It is loaded from a serialized file and used to generate labels for simulated feature vectors. A small amount of Gaussian noise is added to those predictions to form the response variable.

**Compatibility warning:** serialized scikit-learn models can be version-sensitive. Prefer loading with the same scikit-learn version used when the model was saved. A pickle compatibility patch can corrupt model state if the stored tree structure differs, so this notebook first tries a normal load and only gives a clear error if it fails.

In [5]:
try:
    f_hat = joblib.load("data/rf_model.pickle")
    print("Random Forest loaded successfully.")
except Exception as exc:
    raise RuntimeError(
        "Could not load rf_model.pickle. This is commonly caused by a scikit-learn version "
        "mismatch between the environment that saved the model and the current Colab runtime. "
        "Use the original compatible scikit-learn version and reload the file. Original error: "
        f"{type(exc).__name__}: {exc}"
    ) from exc

if not hasattr(f_hat, "predict"):
    raise TypeError("rf_model.pickle did not load into an object with a .predict() method.")

RuntimeError: Could not load rf_model.pickle. This is commonly caused by a scikit-learn version mismatch between the environment that saved the model and the current Colab runtime. Use the original compatible scikit-learn version and reload the file. Original error: FileNotFoundError: [Errno 2] No such file or directory: 'data/rf_model.pickle'

## 3. Load the distribution parameters

The `.npz` file stores the source and target Gaussian parameters. `diff_vec = mean_test - mean_train` defines the mean-shift direction. The target samples are generated using `mean_train + θ × diff_vec` and `cov_test`; therefore, larger \(\theta\) moves the generated mean farther from the source mean along that direction.

In [ ]:
rw_data = np.load('rw_data.npz')
mean_train = rw_data['mean_train']
cov_train = rw_data['cov_train']
mean_test = rw_data['mean_test']
cov_test = rw_data['cov_test']
diff_vec = mean_test - mean_train
sig_y = rw_data['sig_y']

In [ ]:
print("Loaded keys:", list(rw_data.files))
print("mean_train shape:", mean_train.shape)
print("cov_train shape:", cov_train.shape)
print("mean_test shape:", mean_test.shape)
print("cov_test shape:", cov_test.shape)
print("sig_y:", float(np.asarray(sig_y).squeeze()))

In [ ]:
N = 5830
gauss_mv_samples = np.random.multivariate_normal(mean_train, cov_train, size=N)

In [ ]:
x_1 = gauss_mv_samples[:,0:1]
x_2 = gauss_mv_samples[:,1:2]
x_3 = gauss_mv_samples[:,2:3]
x_4 = gauss_mv_samples[:,3:4]
x_5 = gauss_mv_samples[:,4:5]
x_6 = gauss_mv_samples[:,5:6]
x_7 = gauss_mv_samples[:,6:7]
x_8 = gauss_mv_samples[:,7:8]
x_9 = gauss_mv_samples[:,8:9]
x_10 = gauss_mv_samples[:,9:10]
x_11 = gauss_mv_samples[:,10:11]
x_12 = gauss_mv_samples[:,11:12]
x_13 = gauss_mv_samples[:,12:13]
x_14 = gauss_mv_samples[:,13:14]
x_15 = gauss_mv_samples[:,14:15]
x_16 = gauss_mv_samples[:,15:16]
x_17 = gauss_mv_samples[:,16:17]
x_18 = gauss_mv_samples[:,17:18]
x_19 = gauss_mv_samples[:,18:19]
x_20 = gauss_mv_samples[:,19:20]
X_train = np.concatenate([x_1, x_2, x_3, x_4, x_5, x_6, x_7, x_8, x_9, x_10, x_11, x_12, x_13, x_14, x_15, x_16, x_17, x_18, x_19, x_20], axis = 1)

In [ ]:
print("X_train shape:", X_train.shape)
display(pd.DataFrame(X_train, columns=[f"Feature_{i}" for i in range(1, 21)]).head())

In [ ]:
# Validate the loaded Random Forest against the 20-dimensional simulated inputs.
# Avoid mutating private tree internals: compatibility patches can silently change predictions.
expected_features = X_train.shape[1]
actual_features = getattr(f_hat, "n_features_in_", expected_features)
if actual_features != expected_features:
    raise ValueError(
        f"The loaded Random Forest expects {actual_features} features, but the simulation has "
        f"{expected_features}. Check that rf_model.pickle and rw_data.npz belong to the same experiment."
    )

y_train = f_hat.predict(X_train).reshape(-1, 1)
y_train = y_train + np.random.normal(loc=0, scale=float(np.asarray(sig_y).squeeze()), size=y_train.shape)


In [ ]:

# Generate target samples for several shift strengths.
# Keep theta=4 as the main target dataset, matching the final iteration of the original loop.
test_size = 3866
thetas = [1, 2, 3, 4]
X_test_by_theta = {}

for theta in thetas:
    samples = np.random.multivariate_normal(
        mean_train + theta * diff_vec, cov_test, size=test_size
    )
    X_test_by_theta[theta] = samples

X_test = X_test_by_theta[max(thetas)].copy()
feature_names = [f"Feature_{i}" for i in range(1, X_train.shape[1] + 1)]
print(f"Source feature matrix: {X_train.shape}")
print(f"Target feature matrix (theta={max(thetas)}): {X_test.shape}")


In [ ]:
y_test = f_hat.predict(X_test).reshape((test_size,1)) + np.random.normal(loc = 0, scale = sig_y, size = (test_size,1))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the plotting style
sns.set_theme(style="whitegrid")

# Create a figure with subplots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot target distributions (y_train vs y_test)
sns.histplot(y_train.flatten(), kde=True, color='blue', alpha=0.5, label='y_train', ax=axes[0])
sns.histplot(y_test.flatten(), kde=True, color='orange', alpha=0.5, label='y_test', ax=axes[0])
axes[0].set_title('Target Distributions (y_train vs y_test)')
axes[0].set_xlabel('Target Value')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Plot feature distributions (average values across samples for each of the 20 features)
mean_features_train = X_train.mean(axis=0)
mean_features_test = X_test.mean(axis=0)

axes[1].plot(range(1, 21), mean_features_train, marker='o', linestyle='-', color='blue', label='X_train Mean')
axes[1].plot(range(1, 21), mean_features_test, marker='s', linestyle='--', color='orange', label='X_test Mean')
axes[1].set_title('Feature Means (X_train vs X_test)')
axes[1].set_xlabel('Feature Index')
axes[1].set_ylabel('Mean Value')
axes[1].set_xticks(range(1, 21))
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Create a DataFrame for X_test for easier correlation calculation and labeling
feature_names = [f'Feature_{i}' for i in range(1, 21)]
df_test = pd.DataFrame(X_test, columns=feature_names)

# Calculate the Pearson correlation matrix
corr_matrix = df_test.corr()

# Plot the correlation matrix as a heatmap
plt.figure(figsize=(14, 11))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5
)
plt.title('Correlation Matrix of X_test Features', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Create DataFrames for both datasets
feature_names = [f'Feature_{i}' for i in range(1, 21)]
df_train = pd.DataFrame(X_train, columns=feature_names)
df_test = pd.DataFrame(X_test, columns=feature_names)

# Compute correlation matrices
corr_train = df_train.corr()
corr_test = df_test.corr()

# Plot the correlation matrices side-by-side
fig, axes = plt.subplots(1, 2, figsize=(24, 10))

# Train correlation heatmap
sns.heatmap(
    corr_train,
    annot=True,
    fmt=".2f",
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    square=True,
    cbar=True,
    ax=axes[0],
    linewidths=0.5
)
axes[0].set_title('Correlation Matrix of X_train Features', fontsize=16)

# Test correlation heatmap
sns.heatmap(
    corr_test,
    annot=True,
    fmt=".2f",
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    square=True,
    cbar=True,
    ax=axes[1],
    linewidths=0.5
)
axes[1].set_title('Correlation Matrix of X_test Features', fontsize=16)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import scipy.stats

# Extract empirical means and covariances from the generated samples
mu_train = X_train.mean(axis=0)
Sigma_train = np.cov(X_train, rowvar=False)

mu_test = X_test.mean(axis=0)
Sigma_test = np.cov(X_test, rowvar=False)

def multivariate_kl(mu0, Sigma0, mu1, Sigma1):
    """
    Calculate analytical KL divergence KL( N0 || N1 ) between two multivariate Gaussians.
    N0 = N(mu0, Sigma0), N1 = N(mu1, Sigma1)
    """
    k = mu0.shape[0]
    Sigma1_inv = np.linalg.pinv(Sigma1)
    diff = mu1 - mu0

    term1 = np.trace(Sigma1_inv @ Sigma0)
    term2 = diff.T @ Sigma1_inv @ diff
    term3 = -k
    term4 = np.linalg.slogdet(Sigma1)[1] - np.linalg.slogdet(Sigma0)[1]

    kl = 0.5 * (term1 + term2 + term3 + term4)
    return kl

# 1. Calculate overall Multivariate KL Divergence
kl_overall = multivariate_kl(mu_train, Sigma_train, mu_test, Sigma_test)
print(f"Overall Multivariate KL Divergence KL(X_train || X_test): {kl_overall:.4f}\n")

# 2. Calculate Univariate KL Divergence for each feature
print(f"{'Feature':<12} | {'Train Mean':<10} | {'Test Mean':<10} | {'KL Divergence':<12}")
print("-" * 53)

feature_kls = []
for i in range(20):
    m_tr, s_tr = mu_train[i], np.sqrt(Sigma_train[i, i])
    m_te, s_te = mu_test[i], np.sqrt(Sigma_test[i, i])

    # Analytical univariate normal KL divergence formula:
    # KL( N(m_tr, s_tr^2) || N(m_te, s_te^2) )
    kl_feat = np.log(s_te / s_tr) + (s_tr**2 + (m_tr - m_te)**2) / (2 * s_te**2) - 0.5
    feature_kls.append(kl_feat)
    print(f"Feature_{i+1:<5} | {m_tr:<10.3f} | {m_te:<10.3f} | {kl_feat:<12.4f}")

In [ ]:
from scipy.stats import wasserstein_distance
import numpy as np

print(f"{'Feature':<12} | {'Wasserstein Distance':<20}")
print("-" * 38)

feature_w_distances = []
for i in range(20):
    dist = wasserstein_distance(X_train[:, i], X_test[:, i])
    feature_w_distances.append(dist)
    print(f"Feature_{i+1:<5} | {dist:<20.4f}")

# Compute the average univariate Wasserstein distance as a summary metric
avg_w_dist = np.mean(feature_w_distances)
print("-" * 38)
print(f"Average 1D Wasserstein Distance: {avg_w_dist:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")

# Prepare data for plotting
features = [f'F_{i}' for i in range(1, 21)]

plt.figure(figsize=(12, 6))
sns.barplot(x=features, y=feature_w_distances, palette="viridis")

# Add labels and title
plt.title('Univariate Wasserstein Distance per Feature (X_train vs X_test)', fontsize=16)
plt.xlabel('Features', fontsize=12)
plt.ylabel('Wasserstein Distance', fontsize=12)

# Draw a line for the average Wasserstein distance
plt.axhline(avg_w_dist, color='red', linestyle='--', label=f'Average Distance ({avg_w_dist:.4f})')
plt.legend(fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")

# Create a dedicated plot for target distributions
plt.figure(figsize=(10, 6))
sns.histplot(y_train.flatten(), kde=True, color='royalblue', alpha=0.6, label='y_train (Training Targets)', stat='density')
sns.histplot(y_test.flatten(), kde=True, color='darkorange', alpha=0.4, label='y_test (Test Targets)', stat='density')

# Add details
plt.title('Comparison of Target Distributions (y_train vs y_test)', fontsize=16, fontweight='bold')
plt.xlabel('Target Value', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.legend(fontsize=11, loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
import scipy.stats as stats

# Flatten target arrays to ensure 1D inputs
y_tr_flat = y_train.flatten()
y_te_flat = y_test.flatten()

# Perform independent two-sample t-test (Welch's t-test which does not assume equal variance)
t_stat, p_val = stats.ttest_ind(y_tr_flat, y_te_flat, equal_var=False)

print("=== Independent Two-Sample t-test ===")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value:     {p_val:.4e}")

# Interpret results
alpha = 0.05
if p_val < alpha:
    print(f"Result: Reject the null hypothesis at alpha={alpha}. The mean target values differ significantly.")
else:
    print(f"Result: Fail to reject the null hypothesis at alpha={alpha}. There is no statistically significant difference between the means.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

# Calculate Welch-Satterthwaite degrees of freedom
v1 = np.var(y_tr_flat, ddof=1)
v2 = np.var(y_te_flat, ddof=1)
n1 = len(y_tr_flat)
n2 = len(y_te_flat)
df = (v1/n1 + v2/n2)**2 / ((v1/n1)**2 / (n1 - 1) + (v2/n2)**2 / (n2 - 1))

# Define the significance level
alpha = 0.05
t_crit = stats.t.ppf(1 - alpha/2, df)

# Generate x-values for plotting the t-distribution
x = np.linspace(-7.5, 7.5, 1000)
y = stats.t.pdf(x, df)

plt.figure(figsize=(11, 6))
plt.plot(x, y, label=f"Student's t-distribution (df = {df:.1f})", color='blue', linewidth=2)

# Shade the critical/rejection regions (two-tailed)
plt.fill_between(x, y, where=(x <= -t_crit), color='red', alpha=0.3, label=f'Rejection Region (α = {alpha})')
plt.fill_between(x, y, where=(x >= t_crit), color='red', alpha=0.3)

# Draw critical value lines
plt.axvline(-t_crit, color='red', linestyle='--', linewidth=1.5, label=f'Critical Values (±{t_crit:.3f})')
plt.axvline(t_crit, color='red', linestyle='--', linewidth=1.5)

# Draw our observed t-statistic
plt.axvline(t_stat, color='darkgreen', linestyle='-', linewidth=2.5, label=f'Observed t-statistic ({t_stat:.4f})')

# Add text annotations
plt.text(t_stat + 0.2, 0.1, 'Observed t\n(Deep inside rejection region)', color='darkgreen', fontweight='bold', fontsize=10)

# Labels and styling
plt.title("t-test Distribution & Rejection Region", fontsize=15, fontweight='bold')
plt.xlabel("t-value", fontsize=12)
plt.ylabel("Probability Density", fontsize=12)
plt.xlim(-7.5, 7.5)
plt.legend(loc='upper right', fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

## 4. Distribution-shift diagnostics

These plots and statistics characterize the *simulated features and labels*, not model accuracy yet.

- **Target histograms:** compare the response distributions.
- **Feature means and correlation heatmaps:** reveal location shifts and dependence structure.
- **KL divergence:** measures distribution discrepancy under a Gaussian approximation; it is asymmetric and can become unstable for nearly singular covariance matrices.
- **Wasserstein distance:** a one-dimensional distance for each feature; it is easier to interpret in the feature's units, but does not capture all multivariate dependencies.
- **Welch's t-test:** tests equality of the two target means. Statistical significance does not by itself indicate practical importance or prove covariate shift.

## 5. Read tuned hyperparameters and train the benchmark models

The hyperparameter text file is parsed with `ast.literal_eval`, which accepts Python literals without evaluating arbitrary expressions. It must contain the keys used below, including `"Elastic Net"` and `"Artificial Neural Network"`.

In [ ]:

hyper_path = Path("Optim_hyperpar_rf_gaussian.txt")
params_all = ast.literal_eval(hyper_path.read_text(encoding="utf-8"))
params_elnet = params_all["Elastic Net"]

if "Artificial Neural Network" not in params_all:
    raise KeyError("Hyperparameter file must contain 'Artificial Neural Network'.")


In [ ]:
input_shape = (X_train.shape[1],)
neurons_reg = 8
learning_rate = 0.0005
decay_factor = 300
epochs = int(params_all['Artificial Neural Network']['epochs'])
batch_size = int(params_all['Artificial Neural Network']['batch_size'])
neurons_fe = int(params_all['Artificial Neural Network']['neurons_fe'])
regularization = params_all['Artificial Neural Network']['regularization']
dropout = params_all['Artificial Neural Network']['dropout']
steps = X_train.shape[0] // batch_size
if batch_size <= 0 or steps == 0:
    raise ValueError("batch_size must be positive and no larger than the source sample count.")

In [ ]:
neurons_fe

In [ ]:

# Standard ML benchmark: Elastic Net regression
elnet_mod = ElasticNet(**params_elnet, max_iter=max(1000, params_elnet.get("max_iter", 1000)))
elnet_mod.fit(X_train, y_train.ravel())


In [ ]:
# Inspect scikit-learn ElasticNet model parameters
print("=== Elastic Net Model Parameters ===")
print(f"Intercept: {elnet_mod.intercept_[0]:.4f}")
print("\nCoefficients:")
for idx, coef in enumerate(elnet_mod.coef_):
    print(f"  Feature_{idx+1:<2}: {coef:.4f}")

In [ ]:
def feature_extraction(neurons_fe):
    model = Sequential()
    model.add(Dense(neurons_fe, input_shape = input_shape, kernel_regularizer = l2(regularization),
                        activation = "relu", kernel_constraint=MaxNorm(3)))
    model.add(Dropout(dropout))
    return model

In [ ]:
def regression(neurons_reg):
    model = Sequential()
    model.add(Dense(neurons_reg, activation = "relu", kernel_constraint=MaxNorm(3)))
    model.add(Dropout(dropout))
    model.add(Dense(1, activation ="linear"))
    return model

In [ ]:
feature_extractor = feature_extraction(neurons_fe)
regressor = regression(neurons_reg)
inputs = Input(shape = input_shape)
features = feature_extractor(inputs)
outputs = regressor(features)
n_source = X_train.shape[0] // batch_size

In [ ]:
baseline_model = Model(inputs=inputs, outputs=outputs)
baseline_model.compile(optimizer = Adam(learning_rate),
                           loss=["mse"], metrics = ["mse"], run_eagerly=True)

In [ ]:
baseline_model.summary()

In [ ]:

# Neural-network baseline training.
# Record the MSE after each epoch so the learning curve can be inspected.
baseline_epoch_mse = []
for epoch in range(epochs):
    epoch_losses = []
    for _ in range(steps):
        batch_index = np.random.randint(0, n_source)
        start = batch_index * batch_size
        x_batch = X_train[start:start + batch_size]
        y_batch = y_train[start:start + batch_size]
        batch_result = baseline_model.train_on_batch(x_batch, y_batch)
        epoch_losses.append(float(batch_result[0] if isinstance(batch_result, (list, tuple)) else batch_result))
    baseline_epoch_mse.append(float(np.mean(epoch_losses)))
    if (epoch + 1) % max(1, epochs // 10) == 0 or epoch == 0:
        print(f"Epoch {epoch + 1:>4}/{epochs}: mean batch MSE = {baseline_epoch_mse[-1]:.6f}")


In [ ]:

# Predictions from the two non-adaptive benchmark models.
y_pred_elnet = elnet_mod.predict(X_test).reshape(-1, 1)
y_pred_baseline = baseline_model.predict(X_test, verbose=0)


In [ ]:
# Train models: WDGRL
training_ratio = 1
gen_rate = 0.0005
gradient_penalty_weight = 1
was_rate = 0.0005
wd_param = 1

In [ ]:
def wasserstein_loss(y_true, y_pred):
  return K.mean(y_true * y_pred)

def gradient_penalty_loss(y_true, y_pred, averaged_samples, gradient_penalty_weight):
  gradients = K.gradients(y_pred, averaged_samples)[0]
  gradients_sqr = K.square(gradients)
  gradients_sqr_sum = K.sum(gradients_sqr, axis=np.arange(1, len(gradients_sqr.shape)))
  gradient_l2_norm = K.sqrt(gradients_sqr_sum)
  gradient_penalty = gradient_penalty_weight * K.square(1 - gradient_l2_norm)
  return K.mean(gradient_penalty)

def make_wasserstein(neurons_fe):
  model = Sequential()
  model.add(Dense(int(neurons_reg),  input_shape = (int(neurons_fe),)))
  model.add(LeakyReLU())
  model.add(Dense(1))
  return model

In [ ]:
class RandomWeightedAverage(Layer):
  def __init__(self, batch_size):
    super().__init__()
    self.batch_size = batch_size

  def call(self, inputs):
    weights = K.random_uniform((self.batch_size, 1))
    return (weights * inputs[0]) + ((1 - weights) * inputs[1])

In [ ]:
feature_extractor = feature_extraction(neurons_fe)
regressor = regression(neurons_reg)
inputs = Input(shape = input_shape)
features = feature_extractor(inputs)
outputs = regressor(features)

In [ ]:
reg_model = Model(inputs = inputs, outputs = outputs)
reg_model.compile(optimizer = Adam(learning_rate), loss = ["mse"], metrics = ["mse"])


In [ ]:
generator_input_for_wasserstein1 = Input(shape=input_shape)
generator_input_for_wasserstein2 = Input(shape=input_shape)

In [ ]:
generated_samples_for_wasserstein1 = feature_extractor(generator_input_for_wasserstein1)
generated_samples_for_wasserstein2 = feature_extractor(generator_input_for_wasserstein2)

In [ ]:
wasserstein = make_wasserstein(neurons_fe)

In [ ]:
wasserstein_output_from_generator1 = wasserstein(generated_samples_for_wasserstein1)
wasserstein_output_from_generator2 = wasserstein(generated_samples_for_wasserstein2)

In [ ]:
regressor_output_from_generator1 = regressor(generated_samples_for_wasserstein1)

In [ ]:
averaged_samples = RandomWeightedAverage(batch_size)([generated_samples_for_wasserstein2, generated_samples_for_wasserstein1])
averaged_samples_out = wasserstein(averaged_samples)
partial_gp_loss = partial(gradient_penalty_loss, averaged_samples=averaged_samples, gradient_penalty_weight=gradient_penalty_weight)
partial_gp_loss.__name__ = "gradient_penalty"

In [ ]:
for layer in wasserstein.layers:
  layer.trainable = True
wasserstein.trainable = True

for layer in feature_extractor.layers:
  layer.trainable = False
feature_extractor.trainable = False

for layer in regressor.layers:
  layer.trainable = False
regressor.trainable = False

In [ ]:
wasserstein_model = Model(inputs=[generator_input_for_wasserstein2, generator_input_for_wasserstein1], outputs=[wasserstein_output_from_generator2, wasserstein_output_from_generator1, averaged_samples_out])
wasserstein_model.compile(optimizer=Adam(was_rate), loss=[wasserstein_loss, wasserstein_loss, partial_gp_loss])


In [ ]:
for layer in wasserstein.layers:
  layer.trainable = False
wasserstein.trainable = False

for layer in feature_extractor.layers:
  layer.trainable = True
feature_extractor.trainable = True

for layer in regressor.layers:
  layer.trainable = True
regressor.trainable = True

In [ ]:
generator_model = Model(inputs=[generator_input_for_wasserstein2, generator_input_for_wasserstein1], outputs=[regressor_output_from_generator1, wasserstein_output_from_generator2, wasserstein_output_from_generator1])
generator_model.compile(optimizer=Adam(gen_rate), loss=["mse", wasserstein_loss, wasserstein_loss], loss_weights = [1, wd_param, wd_param])


In [ ]:
positive_y = np.ones((batch_size, 1), dtype=np.float32)
negative_y = -positive_y
dummy_y = np.zeros((batch_size, 1), dtype=np.float32)

In [ ]:
n_source = X_train.shape[0] // batch_size
n_target = X_test.shape[0] // batch_size

In [ ]:

# WDGRL adversarial adaptation loop.
# The critic is updated using target/source representation batches, then the representation
# extractor and regressor are updated to reduce source prediction error while aligning domains.
critic_losses = []
generator_losses = []

for epoch in range(epochs):
    epoch_critic, epoch_generator = [], []
    for step in range(steps):
        source_index = np.random.randint(0, n_source)
        target_index = np.random.randint(0, n_target)

        s0, t0 = source_index * batch_size, target_index * batch_size
        x_source_batch = X_train[s0:s0 + batch_size]
        y_source_batch = y_train[s0:s0 + batch_size]
        x_target_batch = X_test[t0:t0 + batch_size]

        critic_result = wasserstein_model.train_on_batch(
            [x_target_batch, x_source_batch],
            [positive_y, negative_y, dummy_y]
        )
        epoch_critic.append(float(critic_result[0] if isinstance(critic_result, (list, tuple)) else critic_result))

        if step % training_ratio == 0:
            gen_result = generator_model.train_on_batch(
                [x_target_batch, x_source_batch],
                [y_source_batch, negative_y, positive_y]
            )
            epoch_generator.append(float(gen_result[0] if isinstance(gen_result, (list, tuple)) else gen_result))

    critic_losses.append(float(np.mean(epoch_critic)))
    generator_losses.append(float(np.mean(epoch_generator)) if epoch_generator else np.nan)
    if (epoch + 1) % max(1, epochs // 10) == 0 or epoch == 0:
        print(f"Epoch {epoch + 1:>4}/{epochs}: critic total loss = {critic_losses[-1]:.6f}, "
              f"generator total loss = {generator_losses[-1]:.6f}")

y_pred_wdgrl = reg_model.predict(X_test, verbose=0)
print("WDGRL training loop finished.")


## 6. Model-behavior visualizations and quantitative evaluation

We compare the models on the same simulated target set. Since `y_test` is generated from the pretrained Random Forest plus random noise, these are **synthetic evaluation results**, not evidence of performance on real-world ground-truth labels.

The following diagnostics include regression metrics, predicted-versus-observed plots, residual distributions, residuals versus predictions, a metric comparison, benchmark training curves, and WDGRL loss curves. Use RMSE/MAE alongside \(R^2\); a negative \(R^2\) means the model is worse than predicting the target mean on this test set.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_true = np.asarray(y_test).reshape(-1)
predictions = {
    "Elastic Net": np.asarray(y_pred_elnet).reshape(-1),
    "Neural Network": np.asarray(y_pred_baseline).reshape(-1),
    "WDGRL": np.asarray(y_pred_wdgrl).reshape(-1),
}

metric_rows = []
for model_name, pred in predictions.items():
    metric_rows.append({
        "Model": model_name,
        "MAE": mean_absolute_error(y_true, pred),
        "RMSE": mean_squared_error(y_true, pred) ** 0.5,
        "R²": r2_score(y_true, pred),
        "Bias (mean residual)": float(np.mean(pred - y_true)),
    })
metrics_df = pd.DataFrame(metric_rows).sort_values("RMSE")
display(metrics_df.style.format({"MAE":"{:.4f}", "RMSE":"{:.4f}", "R²":"{:.4f}", "Bias (mean residual)":"{:.4f}"}))

In [ ]:
# 6.1 Actual versus predicted: closeness to the diagonal indicates better agreement.
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
lims = [min(y_true.min(), *(p.min() for p in predictions.values())),
        max(y_true.max(), *(p.max() for p in predictions.values()))]
for ax, (name, pred) in zip(axes, predictions.items()):
    ax.scatter(y_true, pred, alpha=0.35, s=14)
    ax.plot(lims, lims, "k--", linewidth=1.5, label="Ideal: prediction = actual")
    ax.set(title=name, xlabel="Synthetic target (y_test)", ylabel="Predicted target")
    ax.legend()
fig.suptitle("Predicted versus synthetic target values", y=1.03, fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# 6.2 Residual distributions. Residual = prediction - observed target.
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, pred) in zip(axes, predictions.items()):
    residual = pred - y_true
    sns.histplot(residual, kde=True, stat="density", ax=ax)
    ax.axvline(0, color="black", linestyle="--", linewidth=1.2)
    ax.set(title=f"{name}: residual distribution", xlabel="Residual", ylabel="Density")
plt.tight_layout()
plt.show()

In [ ]:
# 6.3 Residuals versus predictions: patterns/funnels can indicate bias or non-constant variance.
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, pred) in zip(axes, predictions.items()):
    residual = pred - y_true
    ax.scatter(pred, residual, alpha=0.35, s=14)
    ax.axhline(0, color="black", linestyle="--", linewidth=1.2)
    ax.set(title=name, xlabel="Predicted target", ylabel="Residual")
plt.tight_layout()
plt.show()

In [ ]:
# 6.4 Compare scalar evaluation metrics. Lower MAE/RMSE is better; higher R² is better.
long_metrics = metrics_df.melt(id_vars="Model", value_vars=["MAE", "RMSE", "R²"],
                               var_name="Metric", value_name="Score")
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, metric in zip(axes, ["MAE", "RMSE", "R²"]):
    subset = long_metrics[long_metrics["Metric"] == metric]
    sns.barplot(data=subset, x="Model", y="Score", ax=ax)
    ax.set_title(metric)
    ax.tick_params(axis="x", rotation=25)
plt.suptitle("Model comparison on the synthetic target set", y=1.04)
plt.tight_layout()
plt.show()

In [ ]:
# 6.5 Training behavior: neural-network baseline and WDGRL total losses.
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(np.arange(1, len(baseline_epoch_mse) + 1), baseline_epoch_mse, marker=".")
axes[0].set(title="Neural-network baseline: training MSE", xlabel="Epoch", ylabel="Mean batch MSE")
axes[1].plot(np.arange(1, len(critic_losses) + 1), critic_losses, label="Critic total loss")
axes[1].plot(np.arange(1, len(generator_losses) + 1), generator_losses, label="Generator total loss")
axes[1].set(title="WDGRL: total training losses", xlabel="Epoch", ylabel="Total loss")
axes[1].legend()
plt.tight_layout()
plt.show()

print("Interpretation note: adversarial critic/generator losses are not directly comparable to supervised MSE, "
      "and their curves need not decrease monotonically. Check target-set metrics and residual plots as well.")

## 7. Inspect how the covariate shift changes with \(\theta\)

The source distribution is fixed. The target feature means move along `diff_vec` as \(\theta\) increases. This plot visualizes the first two feature dimensions; the full shift is 20-dimensional. These are sampled synthetic distributions, so sample means will not match their theoretical means exactly.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(X_train[:, 0], X_train[:, 1], s=10, alpha=0.2, label="Source")
for theta, samples in X_test_by_theta.items():
    axes[0].scatter(samples[:, 0], samples[:, 1], s=10, alpha=0.18, label=f"Target θ={theta}")
axes[0].set(title="Feature-space shift (first two features)", xlabel="Feature 1", ylabel="Feature 2")
axes[0].legend(markerscale=2, fontsize=8)

theta_means = np.array([X_test_by_theta[t].mean(axis=0) for t in thetas])
for feature_idx in range(min(5, X_train.shape[1])):
    axes[1].plot(thetas, theta_means[:, feature_idx], marker="o", label=f"Feature {feature_idx+1}")
    axes[1].axhline(X_train[:, feature_idx].mean(), linestyle=":", alpha=0.5)
axes[1].set(title="Sample means as shift strength increases", xlabel="Shift strength θ", ylabel="Feature mean")
axes[1].legend()
plt.tight_layout()
plt.show()

## 8. Final interpretation checklist

- Compare models using multiple metrics rather than selecting a model from one plot.
- Examine residuals for systematic overprediction/underprediction and heteroscedasticity.
- WDGRL is intended to learn a representation that is predictive on the source while reducing source/target representation discrepancy. This notebook's metrics can indicate whether it helped on this particular simulation, but do not establish general superiority.
- The target set is sampled from a shifted Gaussian and labels are generated by a supplied Random Forest plus noise. If the original paper defines a different data-generation or evaluation protocol, follow that protocol when reporting results.
- Save the exact dependency versions, random seed, hyperparameters, and model checkpoints if you need reproducible research results.